# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aniqaatiq842-commits/Flyrank-ML-INTERNSHIP/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/aniqaatiq842-commits/Flyrank-ML-INTERNSHIP

Cloning into 'Flyrank-ML-INTERNSHIP'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 136 (delta 46), reused 94 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.91 MiB | 13.28 MiB/s, done.
Resolving deltas: 100% (46/46), done.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
## Finding 1 — The Content Performance Curve

**Paper finding:**  
The paper reports that content performance peaks around 61–90 days, declines after 270 days, and that the 365+ recovery is concentrated in older pages that were refreshed. The paper also states that this should not be interpreted as evidence that age naturally reverses performance decline. :contentReference[oaicite:1]{index=1}

**My methodology question:**  
How is the relationship between content age, refreshing, and performance separated from the timing of the refresh itself? In particular, I would want to understand whether the analysis distinguishes pages that were refreshed because they were already showing signs of decline from pages that were refreshed for other reasons.

**Why this matters:**  
If pages are selected for refresh based on their existing performance or decline, the observed difference after refresh may partly reflect which pages were chosen rather than the refresh action alone. Clarifying the selection and comparison design would help determine how strongly the finding supports a decision to refresh older content.

----------------------------------------------------------------------------------------------------------------------------------------------------------------

## Finding 2 — The Freshness Multiplier

**Paper finding:**  
The paper reports that the 31–90 day freshness window is the strongest stable freshness band and reports a 3.2× health increase and 57× more impressions for 365+ day content refreshed within 30 days. It also notes that the 361+ bucket is unstable because it contains only one declining page. :contentReference[oaicite:2]{index=2}

**My methodology question:**  
What validation or comparison design supports interpreting the difference between recently refreshed and unchanged older pages? In particular, were comparable pages used, and were the groups separated in a way that prevents later performance information from influencing which pages were considered refreshed?

**Why this matters:**  
The size of the reported difference is important, but a large observational difference does not by itself establish that refreshing caused the improvement. A clearer comparison design would help determine whether the measured result is best treated as an observed association or as evidence that supports a stronger intervention claim.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
## 2.1 Original Week-5 split

In Week 5, I used an 80/20 random train-test split with stratification on `trend_direction`. This preserved the class proportions across the two sets and gave a reproducible test split using `random_state=42`.

The Week-5 Decision Tree achieved 0.676 accuracy and 0.6544 macro F1 on this random holdout.

For this validation audit, I will keep the same target, features, model type, and evaluation metrics, but change the split design so that all rows belonging to the same client remain in only one partition. This provides a stricter test of whether the model generalizes across clients.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

DATA_PATH = "Flyrank-ML-INTERNSHIP/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

target = "trend_direction"

features = [
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features]
y = df[target]

print("Dataset shape:", df.shape)
print("X shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

Dataset shape: (30000, 44)
X shape: (30000, 26)
Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [3]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

random_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(
        random_state=42,
        max_depth=5
    ))
])

random_model.fit(X_train_random, y_train_random)

y_pred_random = random_model.predict(X_test_random)

random_accuracy = accuracy_score(y_test_random, y_pred_random)
random_macro_precision = precision_score(
    y_test_random,
    y_pred_random,
    average="macro",
    zero_division=0
)
random_macro_recall = recall_score(
    y_test_random,
    y_pred_random,
    average="macro",
    zero_division=0
)
random_macro_f1 = f1_score(
    y_test_random,
    y_pred_random,
    average="macro",
    zero_division=0
)

print("Original random split results")
print("-----------------------------")
print("Accuracy:", round(random_accuracy, 4))
print("Macro Precision:", round(random_macro_precision, 4))
print("Macro Recall:", round(random_macro_recall, 4))
print("Macro F1:", round(random_macro_f1, 4))

Original random split results
-----------------------------
Accuracy: 0.676
Macro Precision: 0.7284
Macro Recall: 0.6537
Macro F1: 0.6544


## 2.2 Honest client-grouped split

The original random split can place rows from the same client in both training and testing data. For this audit, I use a client-grouped split so that a client appears in only one partition.

This is a stricter generalization test: the model is evaluated on clients it did not see during training. The features, target, model type, and evaluation metrics remain unchanged so that the main change is the validation design.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X, y, groups=groups)
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train_group))
print("Testing rows:", len(X_test_group))

print("Unique training clients:", groups_train.nunique())
print("Unique testing clients:", groups_test.nunique())

print(
    "Clients shared between train and test:",
    len(set(groups_train) & set(groups_test))
)

Training rows: 23837
Testing rows: 6163
Unique training clients: 25
Unique testing clients: 7
Clients shared between train and test: 0


In [5]:
group_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("tree", DecisionTreeClassifier(
        random_state=42,
        max_depth=5
    ))
])

group_model.fit(X_train_group, y_train_group)

y_pred_group = group_model.predict(X_test_group)

group_accuracy = accuracy_score(y_test_group, y_pred_group)

group_macro_precision = precision_score(
    y_test_group,
    y_pred_group,
    average="macro",
    zero_division=0
)

group_macro_recall = recall_score(
    y_test_group,
    y_pred_group,
    average="macro",
    zero_division=0
)

group_macro_f1 = f1_score(
    y_test_group,
    y_pred_group,
    average="macro",
    zero_division=0
)

print("Honest client-grouped split results")
print("------------------------------------")
print("Accuracy:", round(group_accuracy, 4))
print("Macro Precision:", round(group_macro_precision, 4))
print("Macro Recall:", round(group_macro_recall, 4))
print("Macro F1:", round(group_macro_f1, 4))

Honest client-grouped split results
------------------------------------
Accuracy: 0.631
Macro Precision: 0.6776
Macro Recall: 0.6328
Macro F1: 0.6187


In [6]:
comparison = pd.DataFrame({
    "Validation design": [
        "Week-5 random stratified split",
        "Honest client-grouped split"
    ],
    "Accuracy": [
        random_accuracy,
        group_accuracy
    ],
    "Macro Precision": [
        random_macro_precision,
        group_macro_precision
    ],
    "Macro Recall": [
        random_macro_recall,
        group_macro_recall
    ],
    "Macro F1": [
        random_macro_f1,
        group_macro_f1
    ]
})

comparison.round(4)

,Validation design,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Week-5 random stratified split,0.676,0.7284,0.6537,0.6544
1,Honest client-grouped split,0.631,0.6776,0.6328,0.6187


## 2.3 Before vs. after validation comparison

The same Decision Tree configuration was evaluated under two validation designs. The Week-5 result used a random stratified split, while the audit result uses a client-grouped split with no clients shared between training and testing.

The comparison below shows how the measured performance changes when the validation design more closely tests generalization to unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
## 3.1 Feature leakage audit

I reviewed the model features against the prediction target and the intended timing of the prediction.

The main leakage question is whether any feature contains information from the same or a later period used to determine `trend_direction`. A feature can be strongly predictive without being useful for a real decision if it is only known after the outcome has already started to occur.

I therefore classify each feature as:

- **Likely safe:** available before the prediction/outcome window.
- **Timing-sensitive:** potentially valid, but depends on exactly when the feature is measured.
- **Potential leakage:** may directly encode information from the period used to construct the target.


  
    

    
  
    
     
    
  


    
  


| Feature | Leakage status | Reason |
|---|---|---|
| `content_age_days` | Likely safe | Describes content age and can be known at prediction time. |
| `search_volume` | Likely safe | Search-demand metadata can be available before prediction. |
| `competition` | Likely safe | Competition metadata can be available before prediction. |
| `cpc` | Likely safe | CPC is an input/market signal rather than the target itself. |
| `word_count` | Likely safe | Content metadata available before prediction. |
| `char_count` | Likely safe | Content metadata available before prediction. |
| `impressions_90d` | Timing-sensitive | Uses historical performance, but the exact 90-day window must precede the prediction point. |
| `clicks_90d` | Timing-sensitive | Same timing concern as `impressions_90d`. |
| `pageviews_90d` | Timing-sensitive | Must represent historical data only. |
| `sessions_90d` | Timing-sensitive | Must represent historical data only. |
| `users_90d` | Timing-sensitive | Must represent historical data only. |
| `engaged_sessions_90d` | Timing-sensitive | Must represent historical data only. |
| `days_with_impressions` | Timing-sensitive | Depends on the observation window used to calculate it. |
| `days_with_sessions` | Timing-sensitive | Depends on the observation window used to calculate it. |
| `impressions_last_30d` | Potential leakage | Needs to be verified against the period used to construct `trend_direction`. |
| `clicks_last_30d` | Potential leakage | Needs to be verified against the period used to construct `trend_direction`. |
| `sessions_last_30d` | Potential leakage | Needs to be verified against the period used to construct `trend_direction`. |
| `impressions_prev_30d` | Timing-sensitive | Could be safe if it is entirely before the prediction period. |
| `clicks_prev_30d` | Timing-sensitive | Could be safe if it is entirely before the prediction period. |
| `sessions_prev_30d` | Timing-sensitive | Could be safe if it is entirely before the prediction period. |
| `days_since_last_update` | Likely safe | Describes historical update timing. |
| `ctr` | Timing-sensitive | Must be calculated only from data available before prediction. |
| `avg_position` | Timing-sensitive | Must represent the historical observation window, not the future outcome period. |
| `engagement_rate` | Timing-sensitive | Must be calculated from historical data only. |
| `scroll_rate` | Timing-sensitive | Must be calculated from historical data only. |
| `ai_traffic_pct` | Timing-sensitive | Must represent information available before prediction. |

## 3.2 Target construction and leakage risk

The target is `trend_direction`. According to the dataset documentation, trend direction is calculated from the change in impressions between the last 30 days and the previous 30 days.

This creates an important timing question for the model: features describing the same 30-day period used to determine the target may contain direct outcome information.

Therefore, the model's measured performance should be interpreted cautiously unless the feature observation window is confirmed to occur strictly before the target measurement window.

In [7]:
# Inspect the target and relevant time-based features

audit_columns = [
    "trend_direction",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update"
]

df[audit_columns].head(10)

,trend_direction,impressions_last_30d,impressions_prev_30d,clicks_last_30d,clicks_prev_30d,sessions_last_30d,sessions_prev_30d,content_age_days,days_since_last_update
0,down,578,987,2,13,2,9,187,20
1,down,2501,5915,2,1,3,2,445,25
2,down,2382,6089,1,3,1,3,141,20
3,stable,3626,4206,22,17,35,26,463,22
4,down,4211,6452,10,2,14,9,263,14
5,down,617,1009,0,1,4,1,147,20
6,down,1,13,0,0,0,1,90,20
7,stable,636,632,1,0,24,4,445,22
8,down,5696,13828,9,8,36,14,90,20
9,down,252,356,0,0,0,0,257,104


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check whether the target is directly determined by the
# last-30d and previous-30d impression fields.

df[
    [
        "trend_direction",
        "impressions_last_30d",
        "impressions_prev_30d"
    ]
].head(20)


,trend_direction,impressions_last_30d,impressions_prev_30d
0,down,578,987
1,down,2501,5915
2,down,2382,6089
3,stable,3626,4206
4,down,4211,6452
5,down,617,1009
6,down,1,13
7,stable,636,632
8,down,5696,13828
9,down,252,356


## 3.3 Leakage finding

The inspection shows that `impressions_last_30d` and `impressions_prev_30d` vary systematically with `trend_direction`. For example, rows labeled `down` generally show lower recent impressions than the previous 30-day period, while rows labeled `up` show the opposite pattern.

This is expected because the dataset's `trend_direction` is defined using changes in impressions. Therefore, these features are highly related to the target construction.

I cannot conclude from this inspection alone that the features are leakage. The key question is whether these impression windows were fully available at the prediction point or whether they overlap with the period used to construct the target.

For this audit, I therefore classify these features as **timing-sensitive / potential leakage** rather than confirmed leakage.
## 3.4 Leakage audit conclusion

The audit identified a timing risk in several performance features, especially:

- `impressions_last_30d`
- `impressions_prev_30d`
- `clicks_last_30d`
- `clicks_prev_30d`
- `sessions_last_30d`
- `sessions_prev_30d`
- `ctr`
- `avg_position`

These features may be valid decision-time signals if they are calculated entirely from historical information available before the prediction window. However, the current dataset inspection does not by itself establish that separation.

Therefore, I will not claim that the model is leakage-free. A safer statement is:

> **The model was audited for feature leakage, and several historical-performance features were identified as timing-sensitive because they are closely related to the target construction. Their final leakage status depends on confirmation of the feature and target observation windows.**

In [9]:
from sklearn.metrics import confusion_matrix

# Show target distribution
print("Target distribution:")
print(y.value_counts(normalize=True).round(3))

# Compare mean impression ratio by target
audit_df = df[
    [
        "trend_direction",
        "impressions_last_30d",
        "impressions_prev_30d"
    ]
].copy()

audit_df["impression_ratio"] = (
    audit_df["impressions_last_30d"] /
    audit_df["impressions_prev_30d"].replace(0, np.nan)
)

audit_df.groupby("trend_direction")["impression_ratio"].agg(
    ["count", "mean", "median"]
).round(3)

Target distribution:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64


,count,mean,median
trend_direction,,,
down,16262,0.419,0.444
flat,0,NaN,NaN
new,0,NaN,NaN
stable,5962,0.968,0.962
up,4388,2.907,1.625


## 3.5 Leakage audit conclusion

The target distribution is imbalanced, with `down` representing approximately 54.2% of observations, followed by `stable` at 19.9% and `up` at 14.6%. The `new` and `flat` classes are smaller.

The impression-ratio analysis shows a strong directional relationship with the target:

- `down`: median impression ratio = 0.444
- `stable`: median impression ratio = 0.962
- `up`: median impression ratio = 1.625

This pattern is consistent with how `trend_direction` is constructed from changes in impressions. Therefore, the recent and previous impression features are highly informative about the target.

However, this analysis does not by itself prove feature leakage. The key unresolved issue is temporal: whether these feature windows were fully available before the target outcome was defined.

I therefore treat these features as timing-sensitive and avoid claiming that the model is completely leakage-free.

The audit supports a narrower claim: the model's features were reviewed for leakage, and several performance features require explicit confirmation of their observation windows before stronger claims about predictive validity are made.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
## 4.1 Rewriting my model claims

The validation audit changed how I interpret the Week-5 model. The original random split produced higher measured performance than the client-grouped split. Because the grouped split evaluates the model on clients that were not present in training, I use the grouped result as the more conservative evidence for generalization across clients.

I also identified several historical-performance features as timing-sensitive because they are closely related to the construction of `trend_direction`. Therefore, I avoid claims that the model is completely leakage-free or that the identified features cause changes in content performance.


---



---
## 4.2 Original claims vs. safer claims

| Claim that goes too far | Evidence from the audit | Safer claim |
|---|---|---|
| The model accurately predicts content trends. | The model measured 67.6% accuracy on the random split and 63.1% on the client-grouped split. | The model measured 63.1% accuracy under client-grouped validation, with a macro F1 of 0.6187. |
| The model generalizes well to new clients. | The grouped split had 0 clients shared between training and testing, and performance decreased compared with the random split. | The model's performance was measured on unseen clients using a client-grouped validation split. |
| The model can reliably identify declining content. | `down` is the largest target class, but the model was evaluated using multiple classes and the audit found a lower macro F1 under grouped validation. | The model provides directional decision-support for classifying observed content-trend categories, with performance measured using macro F1 and accuracy. |
| The performance features are predictive of decline. | Impression features show a strong relationship with `trend_direction`, but the target itself is constructed from impression changes. | Impression-based features are strongly associated with the observed trend labels, but their temporal relationship to the target requires confirmation. |
| The model is leakage-free. | Several historical-performance features were identified as timing-sensitive. | The feature audit identified no confirmed leakage from the inspection, but several features require confirmation of their observation windows. |
| The model proves which factors cause content decline. | The analysis is observational and model-based rather than causal. | The model identifies patterns that may support content-refresh decision-making; it does not establish causal effects. |

---
## 4.3 Final public-safe claim

Based on the validation audit, the model should be described as a **decision-support model for observed content-trend patterns**, rather than as a causal or fully generalizable predictor.

Under the original random stratified split, the Decision Tree measured 67.6% accuracy and 0.6544 macro F1. Under the stricter client-grouped split, performance measured 63.1% accuracy and 0.6187 macro F1, with no clients shared between training and testing.

The grouped result provides a more conservative measure of performance on unseen clients. The audit also identified several historical-performance features as timing-sensitive because they are closely related to the target construction.

Therefore, the appropriate claim is:

> **The model provides measured, directional decision-support for classifying content trend categories. Its performance was lower under client-grouped validation than under the original random split, and several features require further temporal validation before stronger claims about predictive validity are made.**

This wording reflects what was observed and measured without claiming causation or performance beyond the evidence.





## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
